# Master Cleaning Notebook

This notebook loads the raw trade dataset, applies all shared cleaning steps,
and saves it to be used by all other notebooks.

**Do not run other notebooks without running this one first.**

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# Parameters
input_path = r"C:\Users\fawaz\Desktop\Project\data\Incoming\dummy_2020.csv"
year       = "2020"

In [19]:
# Paths
RAW_PATH      = Path(input_path)
HS_TABLE_PATH = Path(r"C:\Users\fawaz\Desktop\mofne\HSCode_table_updated.xlsx")
OUTPUT_PATH   = Path(r"C:\Users\fawaz\Desktop\Project\outputs") / (RAW_PATH.stem + "_cleaned.csv")

## Step 1 — Load Raw Data

In [20]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Raw shape: {df.shape}")
df.head(2)


Raw shape: (1000, 37)


,item_id,declaration_id,year,customs_office_code,customs_office_name,regime,registration_serial,registration_number,reference_number,registration_date,...,status,specification_code,warehouse_code,exit_office_code,exit_officer_id,operation_name,operation_date,encrypted_declarant_cr,encrypted_consignee_cr,encrypted_exporter_cr
0,2020-2601-8-10000-1,2020-2601-8-10000,2020,2601,Dry Port - Hidd,IM,8,10000,94487,1/8/2020,...,Released,9,9,2601,306,NaN,NaN,CR_HASH_182,CR_HASH_036,NaN
1,2020-2501-2-10001-1,2020-2501-2-10001,2020,2501,Bahrain International Airport,EX,2,10001,56354,3/27/2020,...,Released,2,2,2501,579,NaN,NaN,CR_HASH_093,NaN,CR_HASH_031


## Step 2 — Helper Functions

In [21]:
def clean_hs_12(x):
    """Normalize HS code to exactly 12 digits, right-padded with zeros."""
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = re.sub(r"\D", "", s)
    if len(s) == 0:
        return ""
    return s.ljust(12, "0")[:12]


def to_num(x):
    """Convert a value to float, returning NaN on failure."""
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    if s in ("", "nan", "None", "null", "NULL"):
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def clean_code(x):
    """Uppercase and strip country/regime codes; return NaN for blanks."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s in ("", "NAN", "NONE", "NULL"):
        return np.nan
    if s.endswith(".0"):
        s = s[:-2]
    return s


## Step 3 — Date Parsing

Parse `registration_date` (MM/DD/YYYY) once into `declaration_date`.
Every downstream notebook reads `declaration_date`.
no notebook should re-parse dates from strings.


In [ ]:
# DATE PARSING — canonical date for every downstream notebook
# Handles both formats observed across our datasets:
# Each format is tried explicitly.

raw = df["registration_date"].astype(str).str.strip()

# Try YYYY-MM-DD first (ISO, unambiguous)
parsed = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")

# For anything that failed the first parse, try M/D/YYYY
mask_unparsed = parsed.isna() & raw.notna() & (raw != "nan") & (raw != "")
parsed.loc[mask_unparsed] = pd.to_datetime(
    raw.loc[mask_unparsed], format="%m/%d/%Y", errors="coerce"
)

df["declaration_date"] = parsed

# Sanity check
n_failed = df["declaration_date"].isna().sum()
n_total  = len(df)
pct_failed = 100 * n_failed / n_total

print(f"Date parsing: {n_total - n_failed:,} of {n_total:,} rows parsed cleanly ({pct_failed:.2f}% failed)")

if pct_failed > 1.0:
    print("WARNING: more than 1% of dates failed to parse.")
    print("First 10 unparseable raw values:")
    print(df.loc[df["declaration_date"].isna(), "registration_date"].head(10).tolist())

# Derived fields — kept for convenience, declaration_date is source of truth
df["year"]       = df["declaration_date"].dt.year
df["year_month"] = df["declaration_date"].dt.to_period("M").astype(str)

Date parsing: 1,000 of 1,000 rows parsed cleanly (0.00% failed)


## Step 4 — HS Code Cleaning

In [23]:
df["hs_clean"] = df["hs_code"].apply(clean_hs_12)
df["hs2"]      = df["hs_clean"].str[:2]   # Chapter level (used by CR logic)
df["hs6"]      = df["hs_clean"].str[:6]   # Used by Trade Pattern

print("HS code sample:")
df[["hs_code", "hs_clean", "hs2", "hs6"]].head(5)


HS code sample:


,hs_code,hs_clean,hs2,hs6
0,8471300000,847130000000,84,847130
1,8703221000,870322100000,87,870322
2,3923301000,392330100000,39,392330
3,102211000,102211000000,10,102211
4,1006301000,100630100000,10,100630


## Step 5 — Regime & Trade Type

In [24]:
# Standardize regime to IM / EX only
df["regime"] = df["regime"].astype(str).str.strip().str.upper()
df.loc[~df["regime"].isin(["IM", "EX"]), "regime"] = np.nan

# Re-export flag: only for EXPORTS where origin differs from destination
# goods entered Bahrain from one country, now being exported to another
df["is_reexport"] = (
    (df["regime"] == "EX") &
    df["country_of_origin_code"].notna() &
    df["country_of_destination_code"].notna() &
    (df["country_of_origin_code"] != df["country_of_destination_code"])
)

# Trade type column
df["trade_type"] = np.where(
    df["regime"] == "IM", "Import",
    np.where(df["is_reexport"], "Re-export",
    np.where(df["regime"] == "EX", "Export", None))
)

print(df["trade_type"].value_counts(dropna=False))
print(f"Re-exports: {df['is_reexport'].sum():,}")


trade_type
Import       677
Re-export    277
None          46
Name: count, dtype: int64
Re-exports: 277


## Step 6 — Numeric Columns

In [25]:
df["local_amount_clean"]   = df["local_amount"].apply(to_num)
df["invoice_amount_clean"] = df["invoice_amount"].apply(to_num) if "invoice_amount" in df.columns else np.nan
df["sup_amount_clean"]     = df["sup_amount"].apply(to_num)
df["net_weight_clean"]     = df["net_weight"].apply(to_num)
df["gross_weight_clean"]   = df["gross_weight"].apply(to_num) if "gross_weight" in df.columns else np.nan

# null/zero sup_amount defaults to 1
df["sup_amount_clean"] = df["sup_amount_clean"].replace([0, 0.0], np.nan).fillna(1.0)

print("Numeric nulls:")
print(df[["local_amount_clean", "sup_amount_clean", "net_weight_clean", "gross_weight_clean"]].isna().sum())


Numeric nulls:
local_amount_clean    0
sup_amount_clean      0
net_weight_clean      0
gross_weight_clean    0
dtype: int64


## Step 7 — Country Codes

In [26]:
for col in ["country_of_origin_code", "country_of_origin",
            "country_of_export_code", "country_of_destination_code",
            "country_of_destination"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_code)

# Partner country: who Bahrain is trading with
df["country_of_origin_code"] = df["country_of_origin_code"].fillna(df["country_of_origin"])
df["country_of_destination_code"] = df["country_of_destination_code"].fillna(df["country_of_destination"])

df["partner_country_code"] = np.where(
    df["regime"] == "IM", df["country_of_origin_code"],
    np.where(df["regime"] == "EX", df["country_of_destination_code"], None)
)

df[["country_of_origin_code", "country_of_export_code",
    "country_of_destination_code", "partner_country_code"]].head(3)


,country_of_origin_code,country_of_export_code,country_of_destination_code,partner_country_code
0,CN,CN,BH,CN
1,BH,BH,US,US
2,BH,BH,JP,JP


## Step 8 — Active CR Column 

In [27]:
# For each row, exactly one CR is the Bahrain-side company:
#   Import row  -> encrypted_consignee_cr  (Bahrain importer)
#   Export row  -> encrypted_exporter_cr   (Bahrain exporter)

def get_active_cr(row):
    consignee = row.get("encrypted_consignee_cr")
    exporter  = row.get("encrypted_exporter_cr")
    if pd.notna(consignee) and str(consignee).strip() not in ("", "nan"):
        return str(consignee).strip()
    if pd.notna(exporter) and str(exporter).strip() not in ("", "nan"):
        return str(exporter).strip()
    return np.nan

df["active_cr"] = df.apply(get_active_cr, axis=1)

print(f"Rows with active_cr:    {df['active_cr'].notna().sum()}")
print(f"Rows WITHOUT active_cr: {df['active_cr'].isna().sum()}")
df[["encrypted_consignee_cr", "encrypted_exporter_cr", "active_cr", "trade_type"]].head(5)


Rows with active_cr:    1000
Rows WITHOUT active_cr: 0


,encrypted_consignee_cr,encrypted_exporter_cr,active_cr,trade_type
0,CR_HASH_036,NaN,CR_HASH_036,Import
1,NaN,CR_HASH_031,CR_HASH_031,Re-export
2,NaN,CR_HASH_016,CR_HASH_016,Re-export
3,CR_HASH_036,NaN,CR_HASH_036,Import
4,CR_HASH_046,NaN,CR_HASH_046,Import


## Step 8.5 — HS Description Column (for CR LLM Notebook)

The dataset already contains a `commercial_description` column but some rows have
Arabic text that was corrupted into mojibake during encoding (visible as Ø, Ù characters).

- Use `commercial_description` directly when it is clean English or Arabic text
- Fall back to the HS code lookup table description when the text is garbled or missing
- Store the result in a new column called `hs_desc`

In [28]:
# Load the HS code lookup table and normalize its code column to match hs_clean format
hs_table = pd.read_excel(HS_TABLE_PATH, dtype={"HS Code": str})
hs_table["HS Code"] = hs_table["HS Code"].apply(clean_hs_12)
hs_table = hs_table.drop_duplicates(subset="HS Code")
hs_table = hs_table.rename(columns={"HS Code": "hs_clean", "Description": "hs_desc_lookup"})

print(f"HS table shape: {hs_table.shape}")
hs_table[["hs_clean", "hs_desc_lookup"]].head(3)


HS table shape: (13378, 7)


,hs_clean,hs_desc_lookup
0,101211000010,Of Arab breed males
1,101211000020,Of Arab breed females
2,101219000010,Of non-Arab breed males


In [29]:
# Join the HS lookup table onto the main dataset as a fallback source
df = df.merge(hs_table[["hs_clean", "hs_desc_lookup"]], on="hs_clean", how="left")

print(f"hs_desc_lookup null count after join: {df['hs_desc_lookup'].isna().sum()}")


hs_desc_lookup null count after join: 1000


In [30]:
# Detect garbled Arabic text in the commercial_description column.
# When Arabic UTF-8 text is read with the wrong encoding, it produces
# letters containing Ø and Ù characters

def is_garbled(text):
    if pd.isna(text) or str(text).strip() == "":
        return True
    return any(c in str(text) for c in ["Ø", "Ù", "Ú", "Û", "Ü"])

garbled_mask = df["commercial_description"].apply(is_garbled)

print(f"Clean English descriptions:   {(~garbled_mask).sum()}")
print(f"Garbled or missing:           {garbled_mask.sum()}")


Clean English descriptions:   1000
Garbled or missing:           0


In [31]:
# Build the final hs_desc column.
# Priority: use commercial_description when clean, otherwise fall back to HS table lookup.

df["hs_desc"] = np.where(
    ~garbled_mask,
    df["commercial_description"],
    df["hs_desc_lookup"]
)

# Drop the intermediate lookup column now that hs_desc is built
df = df.drop(columns=["hs_desc_lookup"])

print(f"hs_desc null count:  {df['hs_desc'].isna().sum()}")
print(f"hs_desc fill rate:   {df['hs_desc'].notna().mean() * 100:.2f}%")
df[["hs_clean", "commercial_description", "hs_desc"]].head(10)


hs_desc null count:  0
hs_desc fill rate:   100.00%


,hs_clean,commercial_description,hs_desc
0,847130000000,Portable laptop computers,Portable laptop computers
1,870322100000,Passenger cars 1000-1500cc,Passenger cars 1000-1500cc
2,392330100000,Plastic bottles for packaging,Plastic bottles for packaging
3,102211000000,"Live cattle, pure-bred breeding","Live cattle, pure-bred breeding"
4,100630100000,Long-grain milled rice,Long-grain milled rice
5,901839000000,Medical syringes and needles,Medical syringes and needles
6,610910000000,Cotton t-shirts,Cotton t-shirts
7,100630100000,Long-grain milled rice,Long-grain milled rice
8,102211000000,"Live cattle, pure-bred breeding","Live cattle, pure-bred breeding"
9,102211000000,"Live cattle, pure-bred breeding","Live cattle, pure-bred breeding"


## Step 9 — Final Check & Save

In [32]:
print("Final cleaned shape:", df.shape)
print()
print("Key column null counts:")
key_cols = [
    "hs_clean", "hs2", "hs6", "hs_desc",
    "declaration_date", "year", "year_month",
    "regime", "trade_type", "is_reexport",
]
print(df[key_cols].isna().sum())

# Verify declaration_date is datetime, not string
print(f"\ndeclaration_date dtype: {df['declaration_date'].dtype}")
assert df['declaration_date'].dtype.kind == 'M', "declaration_date must be datetime64 — check Step 3"


Final cleaned shape: (1000, 52)

Key column null counts:
hs_clean             0
hs2                  0
hs6                  0
hs_desc              0
declaration_date     0
year                 0
year_month           0
regime              46
trade_type          46
is_reexport          0
dtype: int64

declaration_date dtype: datetime64[ns]


In [33]:
# Saves the cleaned data into a cumulative file to keep the thereshold for future runs
CUMULATIVE_PATH = Path(r"C:\Users\fawaz\Desktop\Project\outputs\cumulative_cleaned.csv")

# 1) Save this file's cleaned output
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved per-file cleaned output: {OUTPUT_PATH}")
print(f"  Shape: {df.shape}")

# 2) Append to cumulative cleaned file (deduplicate by item_id)
if CUMULATIVE_PATH.exists():
    existing = pd.read_csv(CUMULATIVE_PATH, low_memory=False, parse_dates=["declaration_date"])
    print(f"  Existing cumulative: {existing.shape[0]:,} rows")
    combined = pd.concat([existing, df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["item_id"], keep="last")
    combined = combined.sort_values("declaration_date").reset_index(drop=True)
    print(f"  After dedup: {combined.shape[0]:,} rows ({combined.shape[0] - existing.shape[0]:+,} new)")
else:
    combined = df.copy()
    print(f"  No existing cumulative file — creating fresh with {len(combined):,} rows")

combined.to_csv(CUMULATIVE_PATH, index=False)
print(f"  Saved cumulative: {CUMULATIVE_PATH}")
print(f"  Total rows: {len(combined):,}")
print(f"  Date range: {combined["declaration_date"].min()} to {combined["declaration_date"].max()}")

# The cumulative file will have a high amount of rows and running the anomaly_pipeline.ipynb will take a long time


Saved per-file cleaned output: C:\Users\fawaz\Desktop\Project\outputs\dummy_2020_cleaned.csv
  Shape: (1000, 52)
  Existing cumulative: 1,000 rows
  After dedup: 1,000 rows (+0 new)
  Saved cumulative: C:\Users\fawaz\Desktop\Project\outputs\cumulative_cleaned.csv
  Total rows: 1,000
  Date range: 2020-01-01 00:00:00 to 2020-12-30 00:00:00
